# RF-DETR人物追跡からカービィアニメーションを生成（コンパクトColab版）

NotebookとランタイムZIPを分離し、Notebook本体を小さく保ちます。
最初のセルで検証済みランタイムZIPを1個選択し、その後に処理対象動画を選択します。


## Colab初期化

`rf-detr-kirby-colab-bundle-v7.zip` を選択します。パス区切りをLinux形式へ正規化してから展開します。


In [ ]:
#@title Upload and initialize the Kirby runtime bundle
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

COLAB_BOOTSTRAP_VERSION = '2026-07-12.compact-7'
print(f'Kirby Colab bootstrap: {COLAB_BOOTSTRAP_VERSION}')
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import files

    print('rf-detr-kirby-colab-bundle-v7.zip を選択してください。')
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(zip_names) != 1:
        raise FileNotFoundError(f'ZIPを1個だけ選択してください。選択内容: {list(uploaded)}')

    bundle_name = zip_names[0]
    bundle_path = Path('/content') / bundle_name
    bundle_path.write_bytes(uploaded[bundle_name])
    if not zipfile.is_zipfile(bundle_path):
        raise zipfile.BadZipFile(f'有効なZIPではありません: {bundle_name}')

    extract_root = Path('/content/kirby-rfdetr-upload')
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True)

    with zipfile.ZipFile(bundle_path) as archive:
        members = [member for member in archive.infolist() if not member.is_dir()]
        for index, member in enumerate(members, 1):
            normalized_name = member.filename.replace(chr(92), '/')
            relative = Path(*[part for part in normalized_name.split('/') if part])
            if relative.is_absolute() or '..' in relative.parts:
                raise ValueError(f'安全でないZIPパス: {member.filename}')
            target = extract_root / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open('wb') as destination:
                shutil.copyfileobj(source, destination)
            if index % 100 == 0 or index == len(members):
                print(f'展開: {index}/{len(members)}')

    package_dirs = sorted(
        path for path in extract_root.rglob('rfdetr_demo')
        if path.is_dir() and path.parent.name == 'src'
    )
    if not package_dirs:
        sample = [str(path.relative_to(extract_root)) for path in extract_root.rglob('*') if path.is_file()][:30]
        raise FileNotFoundError(
            '展開後に src/rfdetr_demo がありません。ZIP内容先頭: ' + repr(sample)
        )

    COLAB_ROOT = package_dirs[0].parent.parent
    required = [
        COLAB_ROOT / 'src' / 'rfdetr_demo' / 'animation' / 'puppet_video.py',
        COLAB_ROOT / 'カービィ.png',
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError('ZIPの必須ファイルが不足しています: ' + repr(missing))

    runtime_dependencies = [
        'requests', 'tqdm', 'pyyaml', 'scipy', 'pydantic>=2,<3',
        'transformers>=5.1.0,<6.0.0', 'supervision>=0.29.0',
        'pyDeprecate>=0.9,<0.10', 'opencv-python-headless>=4.8',
    ]
    subprocess.check_call(
        [
            sys.executable,
            '-m',
            'pip',
            'install',
            '--upgrade-strategy',
            'only-if-needed',
            *runtime_dependencies,
        ]
    )
    sys.path.insert(0, str(COLAB_ROOT / 'src'))
    os.environ['RFDETR_KIRBY_ROOT'] = str(COLAB_ROOT)
    print('Colab workspace:', COLAB_ROOT)
    print('rfdetr_demo:', COLAB_ROOT / 'src' / 'rfdetr_demo')
else:
    print('ローカルJupyterとして実行します。')


## 1. パスと処理設定

最初は `QUICK_TEST = True` で短時間確認し、正常なら `False` にして全編を生成します。`FRAME_STRIDE` を大きくすると推論は速くなり、完成動画は元fpsへ補間されます。

In [ ]:
ROOT = Path(os.environ.get('RFDETR_KIRBY_ROOT', Path.cwd())).resolve()
if not (ROOT / 'src' / 'rfdetr_demo').exists():
    candidates = [ROOT, *ROOT.parents]
    ROOT = next((p for p in candidates if (p / 'src' / 'rfdetr_demo').exists()), ROOT)

SOURCE_VIDEO = Path(os.environ.get('KIRBY_SOURCE_VIDEO', ROOT / 'sample' / 'mzoo.mov'))
KIRBY_IMAGE = ROOT / 'カービィ.png'
MODEL_DIR = ROOT / 'artifacts' / 'models'
MODEL_WEIGHT = MODEL_DIR / 'rf-detr-keypoint-preview-xlarge.pth'
OUTPUT_DIR = ROOT / 'artifacts' / 'kirby_notebook'
RIG_DIR = OUTPUT_DIR / 'rig'
CONTROLS_JSON = OUTPUT_DIR / 'controls.json'
TRACKING_OVERLAY = OUTPUT_DIR / 'tracking_overlay.mp4'
KIRBY_VIDEO = OUTPUT_DIR / 'kirby_tracked.mp4'

QUICK_TEST = True
FRAME_STRIDE = 6
MAX_INFERENCE_FRAMES = 30 if QUICK_TEST else None
DETECTION_THRESHOLD = 0.25
KEYPOINT_THRESHOLD = 0.15
TRACK_ID = None  # Noneなら最初のフレームにいる人物から自動選択

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / 'src'))
os.environ['RF_HOME'] = str(MODEL_DIR)

print('repo:', ROOT)
print('source:', SOURCE_VIDEO)
print('output:', OUTPUT_DIR)

## 2. 必須ファイルを検査

不足しているファイルを推測で進めず、ここで明示的に停止します。

In [ ]:
if not KIRBY_IMAGE.exists() and (Path('/content') / KIRBY_IMAGE.name).exists():
    shutil.copy2(Path('/content') / KIRBY_IMAGE.name, KIRBY_IMAGE)
    print('カービィ画像を /content から復元しました:', KIRBY_IMAGE)

if IN_COLAB and not KIRBY_IMAGE.exists():
    print('カービィ画像PNGを選択してください。')
    uploaded = files.upload()
    image_name = next((name for name in uploaded if Path(name).suffix.lower() in {'.png', '.webp'}), None)
    if image_name is None:
        raise FileNotFoundError('カービィ画像PNGが選択されていません。')
    KIRBY_IMAGE = ROOT / image_name
    KIRBY_IMAGE.write_bytes(uploaded[image_name])

if IN_COLAB and not SOURCE_VIDEO.exists():
    print('追跡したい動画を選択してください。')
    uploaded = files.upload()
    video_name = next((name for name in uploaded if Path(name).suffix.lower() in {'.mp4', '.mov', '.avi', '.mkv'}), None)
    if video_name is None:
        raise FileNotFoundError('動画ファイルが選択されていません。')
    SOURCE_VIDEO = ROOT / video_name
    SOURCE_VIDEO.write_bytes(uploaded[video_name])
    print('uploaded video:', SOURCE_VIDEO)

if not MODEL_WEIGHT.exists():
    from urllib.request import urlretrieve
    from tqdm.auto import tqdm

    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    print('RF-DETRキーポイント重みを取得します。')
    with tqdm(unit='B', unit_scale=True, desc='モデル重み') as progress:
        downloaded = [0]
        def reporthook(block_count, block_size, total_size):
            if total_size > 0:
                progress.total = total_size
            current = block_count * block_size
            progress.update(max(0, current - downloaded[0]))
            downloaded[0] = current
        urlretrieve(
            'https://storage.googleapis.com/rfdetr/rf-detr-keypoint-preview-xlarge.pth',
            MODEL_WEIGHT,
            reporthook,
        )

In [ ]:
required = {
    '入力動画': SOURCE_VIDEO,
    'カービィ画像': KIRBY_IMAGE,
    'RF-DETRキーポイント重み': MODEL_WEIGHT,
}
missing = {name: path for name, path in required.items() if not path.exists()}
for name, path in required.items():
    state = 'OK' if path.exists() else 'MISSING'
    size = f'{path.stat().st_size / 1024 / 1024:.1f} MB' if path.exists() else '-'
    print(f'{state:7} {name:24} {size:>10}  {path}')
if missing:
    raise FileNotFoundError('不足ファイル: ' + ', '.join(f'{k}={v}' for k, v in missing.items()))

## 3. 入力動画を確認

In [ ]:
import cv2
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from urllib.parse import quote

capture = cv2.VideoCapture(str(SOURCE_VIDEO))
video_info = {
    'frames': int(capture.get(cv2.CAP_PROP_FRAME_COUNT)),
    'fps': float(capture.get(cv2.CAP_PROP_FPS)),
    'width': int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)),
    'height': int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT)),
}
capture.release()
video_info['duration_sec'] = video_info['frames'] / max(video_info['fps'], 1.0)
print(video_info)
PREVIEW_VIDEO = OUTPUT_DIR / 'input_preview_h264.mp4'
preview_ready = PREVIEW_VIDEO.exists() and PREVIEW_VIDEO.stat().st_mtime >= SOURCE_VIDEO.stat().st_mtime
if not preview_ready:
    print('ブラウザ再生用H.264プレビューを作成します。')
    command = [
        'ffmpeg', '-y', '-i', str(SOURCE_VIDEO), '-vf',
        'scale=-2:960:force_original_aspect_ratio=decrease',
        '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '23', '-pix_fmt', 'yuv420p',
        '-an', '-movflags', '+faststart', str(PREVIEW_VIDEO),
    ]
    result = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    preview_ready = result.returncode == 0 and PREVIEW_VIDEO.exists()
    if not preview_ready:
        print('H.264変換に失敗しました。先頭フレームを表示して処理を続行します。')
        print(result.stderr[-1000:])

if preview_ready:
    preview_url = '/files/' + quote(str(PREVIEW_VIDEO))
    display(HTML(f'<video controls width=480 src={preview_url}></video>'))
else:
    capture = cv2.VideoCapture(str(SOURCE_VIDEO))
    ok, first_frame = capture.read()
    capture.release()
    if not ok:
        raise RuntimeError('OpenCVで先頭フレームを取得できません。動画を再エンコードしてください。')
    plt.figure(figsize=(5, 9))
    plt.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()

## 4. カービィ連続メッシュリグを生成

画像は分割しません。目と頬を保護し、大口時には目を少し上へ移動・縮小する表情プロファイルもリグへ保存します。

In [ ]:
import json
from PIL import Image

def prepare_kirby_mesh_rig(source, output_dir, padding_ratio=0.25):
    image = Image.open(source).convert('RGBA')
    source_width, source_height = image.size
    padding_x = round(source_width * padding_ratio)
    padding_y = round(source_height * padding_ratio)
    width = source_width + padding_x * 2
    height = source_height + padding_y * 2
    width += width % 2
    height += height % 2
    canvas = Image.new('RGBA', (width, height), (0, 0, 0, 0))
    canvas.paste(image, (padding_x, padding_y))
    output_dir.mkdir(parents=True, exist_ok=True)
    canvas.save(output_dir / 'full_body.png')
    def point(x, y):
        return [(padding_x + source_width * x) / width, (padding_y + source_height * y) / height]
    def radius(x, y):
        return [source_width * x / width, source_height * y / height]
    pivots = {'Body': (0.50, 0.68), 'Head': (0.50, 0.66), 'Left Arm': (0.16, 0.52),
              'Right Arm': (0.84, 0.52), 'Left Leg': (0.33, 0.78), 'Right Leg': (0.67, 0.78)}
    manifest = {
        'source': str(source), 'canvas': {'width': width, 'height': height},
        'render_mode': 'continuous_mesh', 'use_residual_layer': False,
        'full_body': {'file': 'full_body.png', 'bbox': [0, 0, width, height]},
        'mesh_profile': {'head_cutoff': 0.86, 'head_blend': 0.38, 'head_angle_gain': 0.65,
            'torso_radius_x': 0.42, 'torso_radius_y': 0.38, 'body_angle_gain': 0.60,
            'spine_curve_gain': 0.45, 'arm_radius_x': 0.28, 'arm_radius_y': 0.30,
            'arm_lower_bias': 0.28, 'arm_angle_gain': 0.70, 'leg_radius_x': 0.23,
            'leg_radius_y': 0.25, 'leg_lower_bias': 0.60, 'leg_angle_gain': 0.60},
        'face_profile': {'protect_shape': 'ellipse', 'center': point(0.50, 0.39),
            'radius': radius(0.25, 0.29), 'feather': 0.22,
            'mouth_reaction': {'eye_lift': source_height * 0.020 / height, 'eye_shrink': 0.055},
            'protected_features': [
                {'name': 'left_eye', 'center': point(0.41, 0.37), 'radius': radius(0.075, 0.19)},
                {'name': 'right_eye', 'center': point(0.59, 0.37), 'radius': radius(0.075, 0.19)},
                {'name': 'left_cheek', 'center': point(0.28, 0.47), 'radius': radius(0.12, 0.10)},
                {'name': 'right_cheek', 'center': point(0.72, 0.47), 'radius': radius(0.12, 0.10)}],
            'mouth': {'center': point(0.50, 0.55), 'cover_radius': radius(0.115, 0.095),
                'size_scale': [source_width / width, source_height / height],
                'outline_color_rgba': [218, 31, 112, 255], 'mouth_color_rgba': [91, 25, 55, 255],
                'tongue_color_rgba': [247, 89, 145, 255]}},
        'parts': [{'name': name, 'z': index,
                   'pivot': [padding_x + x * source_width, padding_y + y * source_height],
                   'center': [padding_x + x * source_width, padding_y + y * source_height]}
                  for index, (name, (x, y)) in enumerate(pivots.items())],
    }
    (output_dir / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    return manifest

manifest = prepare_kirby_mesh_rig(KIRBY_IMAGE, RIG_DIR)
print('canvas:', manifest['canvas'])
print('rig:', RIG_DIR)

## 5. RF-DETRで人物を検出・追跡

GPUが利用可能なら自動的に使用されます。CPUでは長時間かかるため、まず短時間テストを推奨します。

In [ ]:
import math
import sys
import torch
from tqdm.auto import tqdm

source_candidates = [
    ROOT,
    *(path.parent.parent for path in Path('/content').glob('**/src/rfdetr_demo')),
    Path('/content/rf-detr'),
    Path('/content'),
]
source_root = next(
    (path for path in source_candidates if (path / 'src' / 'rfdetr_demo').exists()),
    None,
)
if source_root is None:
    raise FileNotFoundError('rfdetr_demo が見つかりません。v7バンドルを再アップロードしてください。')
sys.path.insert(0, str(source_root / 'src'))
print('RF-DETR source:', source_root / 'src')
from rfdetr_demo.inference.models import build_keypoint_model
from rfdetr_demo.animation.video_export import run_torso_animation_export

print('torch:', torch.__version__)
print('cuda:', torch.cuda.is_available())
model = build_keypoint_model()
expected_inference_frames = math.ceil(video_info['frames'] / FRAME_STRIDE)
if MAX_INFERENCE_FRAMES is not None:
    expected_inference_frames = min(expected_inference_frames, MAX_INFERENCE_FRAMES)
with tqdm(total=expected_inference_frames, desc='RF-DETR検出・追跡', unit='frames') as progress:
    def export_progress(completed, total):
        progress.total = total or progress.total
        progress.update(max(0, completed - progress.n))
    export_summary = run_torso_animation_export(
        source_path=SOURCE_VIDEO,
        json_path=CONTROLS_JSON,
        overlay_path=TRACKING_OVERLAY,
        threshold=DETECTION_THRESHOLD,
        keypoint_threshold=KEYPOINT_THRESHOLD,
        frame_stride=FRAME_STRIDE,
        max_frames=MAX_INFERENCE_FRAMES,
        model=model,
        progress_callback=export_progress,
    )
export_summary

## 6. 追跡人物を選択

自動選択では、先頭検出フレームに存在し、全体で最も多く観測された追跡IDを使用します。別人物を使う場合は最初の設定セルで `TRACK_ID` を指定してください。

In [ ]:
import json
from collections import Counter

controls = json.loads(CONTROLS_JSON.read_text(encoding='utf-8'))
frames = controls['frames']
counts = Counter(
    int(person['track_id'])
    for frame in frames
    for person in frame.get('people', [])
    if not person.get('is_ghost', False)
)
first_ids = [int(p['track_id']) for p in frames[0].get('people', []) if not p.get('is_ghost', False)]
if TRACK_ID is None:
    if not first_ids:
        raise RuntimeError('先頭検出フレームに追跡人物がいません。しきい値を下げて再実行してください。')
    selected_track_id = max(first_ids, key=lambda track_id: counts[track_id])
else:
    selected_track_id = int(TRACK_ID)
    if selected_track_id not in first_ids:
        raise ValueError(f'Track {selected_track_id} は先頭検出フレームに存在しません: {first_ids}')
print('track counts:', dict(counts))
print('selected track:', selected_track_id)
print('complete source:', controls.get('complete_source'))

## 7. カービィ動画を生成

`QUICK_TEST=True` の結果は検出した範囲だけの確認動画です。全編出力時は設定を `False` に戻し、手順5以降を再実行してください。

In [ ]:
from rfdetr_demo.animation.puppet_video import render_puppet_video
from tqdm.auto import tqdm

with tqdm(desc='カービィレンダリング', unit='frames') as progress:
    def render_progress(completed, total):
        progress.total = total
        progress.update(max(0, completed - progress.n))
    render_summary = render_puppet_video(
        controls_json=CONTROLS_JSON,
        rig_dir=RIG_DIR,
        output_path=KIRBY_VIDEO,
        track_id=selected_track_id,
        dynamics_enabled=True,
        resample_to_source_fps=True,
        progress_callback=render_progress,
    )
render_summary

## 8. 結果をプレビュー

In [ ]:
import subprocess
from pathlib import Path

from IPython.display import Video, display


def to_browser_playable(source: Path) -> Path:
    target = source.with_name(f'{source.stem}_h264.mp4')
    if target.exists() and target.stat().st_mtime >= source.stat().st_mtime:
        return target
    command = [
        'ffmpeg', '-y', '-i', str(source),
        '-c:v', 'libx264', '-preset', 'veryfast', '-crf', '20',
        '-pix_fmt', 'yuv420p', '-movflags', '+faststart', str(target),
    ]
    result = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    if result.returncode != 0 or not target.exists():
        raise RuntimeError('ブラウザ再生用H.264変換に失敗しました: ' + result.stderr[-1000:])
    return target


print('追跡オーバーレイ:', TRACKING_OVERLAY)
display(Video(str(to_browser_playable(TRACKING_OVERLAY)), width=720, embed=IN_COLAB))
print('カービィ動画:', KIRBY_VIDEO)
display(Video(str(to_browser_playable(KIRBY_VIDEO)), width=560, embed=IN_COLAB))